# 作业1：Data Types, Functions and Conditionals

**姓名：** 邵鹏祥
**学号：** 20244080303
**提交文件：** 20244080303_邵鹏祥_hw01.ipynb

本作业共四个部分：热身（定义简单函数）、欧几里得算法、近似欧拉数 e、测试整数属性。
以下代码按题目要求的顺序依次存放与运行，输出结果已保存在本笔记本中。

## 学号信息

按注意事项第 3 条，先将学号修改为自己的学号，再按顺序运行各个代码框。

In [1]:
import os, uuid, hashlib, platform, getpass

# ===== 1. 填写学号 =====
STUDENT_ID = 20244080303  #修改为自己的学号

# ===== 2. 打印 conda 环境 =====
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

# ===== 3. 机器指纹（绑定当前电脑）=====
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

# ===== 4. 生成唯一 ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]     {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] base
[学号]     20244080303
[机器指纹] 5677f62d0c129bf5
[RUN_ID]   20244080303-5677f62d0c129bf5


---

# 一、热身：定义简单函数

1. `say_hi`：不接收参数，打印 `"Hello, world!"`。
2. `goat_pad`：接收一个字符串，打印该字符串并在前后都加上 `"goat"`。
3. `print_n`：接收一个字符串 `s` 和一个整数 `n`，把 `s` 打印 `n` 次，每次单独一行。

In [2]:
def say_hi():
    """不接收参数，打印问候语 "Hello, world!"。"""
    print("Hello, world!")


def goat_pad(s):
    """接收一个字符串 s，打印 s 并在前后都加上字符串 "goat"。"""
    print("goat" + s + "goat")


def print_n(s, n):
    """把字符串 s 打印 n 次，每次单独占一行。"""
    for _ in range(n):
        print(s)

In [3]:
# 简单测试
say_hi()
goat_pad("hello")
print_n("Echo!", 3)

Hello, world!
goathellogoat
Echo!
Echo!
Echo!


---

# 二、欧几里得算法

欧几里得算法（辗转相除法）用于求两个数 `m` 和 `n` 的最大公约数（GCD）：反复用较小数
除较大数并取余数，直到余数为 0，此时最后一个非零余数就是最大公约数。

### 1. 实现 `gcd` 函数

In [4]:
def gcd(m, n):
    """用欧几里得算法（辗转相除法）返回整数 m 和 n 的最大公约数。"""
    while n != 0:
        m, n = n, m % n
    return m

### 2. 计算题目给出的三组数的 GCD

In [5]:
pairs = [(2018, 2019), (1200, 300), (5040, 60)]
for m, n in pairs:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")

gcd(2018, 2019) = 1
gcd(1200, 300) = 300
gcd(5040, 60) = 60


### 3. 如果一个或两个参数是负数，函数会做什么？这种行为有意义吗？

先用几组负数实际验证一下：

In [6]:
for m, n in [(-2018, 2019), (2018, -2019), (-2018, -2019), (-1200, 300), (1200, -300)]:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")

gcd(-2018, 2019) = 1
gcd(2018, -2019) = -1
gcd(-2018, -2019) = -1
gcd(-1200, 300) = 300
gcd(1200, -300) = -300


**回答：**

Python 中 `%` 运算的结果符号跟随除数（`n`），因此上面的 `gcd` 在出现负数时行为是：

| 参数情况 | 返回值 | 说明 |
| --- | --- | --- |
| 两个参数都为正 | 正的 GCD | 正常 |
| 只有第二个参数 `n` 为负 | 负的 GCD | 结果符号跟着 `n` |
| 只有第一个参数 `m` 为负 | 正的 GCD | 中间取余时 `n` 变为正数，最终结果为正 |

也就是说，`gcd(2018, -2019)` 会返回 `-1`，而不是 `1`。

这种**行为在数学上并不合理**：最大公约数定义在整数的绝对值上，按定义它是一个非负整数
（`gcd(m, n) = gcd(|m|, |n|) ≥ 0`），而上面的实现可能返回负数，仅仅是 Python 取模
运算符号规则的副作用，属于实现细节而非数学结论。

修正办法是在函数入口（或返回处）取绝对值，例如 `m, n = abs(m), abs(n)`，这样对任意非零
整数输入都能得到非负的最大公约数（Python 标准库的 `math.gcd` 也只接受并返回非负结果）。

---

# 三、近似欧拉数 e

自然对数的底数 e 的无限和为

$$e = \sum_{k=0}^{\infty} \frac{1}{k!} = 1 + \frac{1}{1} + \frac{1}{2} + \frac{1}{6} + \frac{1}{24} + \cdots$$

其中 `k!` 表示 `k` 的阶乘，约定 `0! = 1`。

### 1. 函数 `euler_limit`

由伯努利给出的早期特征：当 $x \to \infty$ 时，$\left(1 + \frac{1}{x}\right)^x \to e$。
取 $x = n$ 即可近似 e。

In [7]:
import math


def euler_limit(n):
    """用 (1 + 1/n) ** n 近似 e，返回浮点数。"""
    return (1 + 1 / n) ** n

### 2. 函数 `euler_infinite_sum`

返回公式 (1) 中**前 n 项**之和。注意公式中的和从 `k = 0` 开始计数（0-索引），而本函数
从 `n = 1` 开始计数（1-索引）：`euler_infinite_sum(0)` 返回 0，
`euler_infinite_sum(1)` 返回 1，`euler_infinite_sum(4)` 返回 `1 + 1 + 1/2 + 1/6 ≈ 2.667`。

In [8]:
def euler_infinite_sum(n):
    """返回 e = sum(1/k!) 的前 n 项之和（n 从 1 开始计数），返回浮点数。"""
    total = 0.0
    factorial = 1.0
    for k in range(n):          # k = 0, 1, ..., n-1，正好是前 n 项
        if k > 0:
            factorial *= k      # 依次得到 0! = 1, 1! = 1, 2! = 2, 3! = 6 ...
        total += 1.0 / factorial
    return total

### 3. 函数 `euler_approx`

用公式 (1) 的和不断累加，直到近似值与真实值 `math.exp(1)` 的差小于 $\epsilon$ 为止。

In [9]:
def euler_approx(eps):
    """用公式 (1) 累加求 e 的近似值，使其与 math.exp(1) 的误差小于 eps。"""
    true_e = math.exp(1)
    total = 0.0
    factorial = 1.0
    k = 0
    while abs(total - true_e) >= eps:   # 误差还不够小，继续加下一项
        if k > 0:
            factorial *= k
        total += 1.0 / factorial
        k += 1
    return total

### 4. 函数 `print_euler_sum_table`

打印 `euler_infinite_sum(k)` 的值，`k` 从 1 到 `n`，每行一个。

In [10]:
def print_euler_sum_table(n):
    """打印 euler_infinite_sum(k) 的值，k 从 1 到 n，每行一个。"""
    for k in range(1, n + 1):
        print(euler_infinite_sum(k))

### 测试上述四个函数

In [11]:
print("e 的真实值 math.exp(1)         =", math.exp(1))
print("euler_infinite_sum(4)          =", euler_infinite_sum(4))
print("euler_infinite_sum(0)          =", euler_infinite_sum(0))
print("euler_infinite_sum(1)          =", euler_infinite_sum(1))
print("euler_limit(1)                 =", euler_limit(1))
print("euler_limit(10)                =", euler_limit(10))
print("euler_limit(1000)              =", euler_limit(1000))
print("euler_limit(1000000)           =", euler_limit(1000000))
print("euler_approx(1e-3)             =", euler_approx(1e-3))
print("euler_approx(1e-10)            =", euler_approx(1e-10))

e 的真实值 math.exp(1)         = 2.718281828459045
euler_infinite_sum(4)          = 2.6666666666666665
euler_infinite_sum(0)          = 0.0
euler_infinite_sum(1)          = 1.0
euler_limit(1)                 = 2.0
euler_limit(10)                = 2.5937424601000023
euler_limit(1000)              = 2.7169239322355936
euler_limit(1000000)           = 2.7182804690957534
euler_approx(1e-3)             = 2.7180555555555554
euler_approx(1e-10)            = 2.7182818284467594


In [12]:
# 打印前 10 项部分和构成的表
print_euler_sum_table(10)

1.0
2.0
2.5
2.6666666666666665
2.708333333333333
2.7166666666666663
2.7180555555555554
2.7182539682539684
2.71827876984127
2.7182815255731922


---

# 四、测试整数属性

一个正整数 `n` 是 2 的幂，如果 $n = 2^p$ 对某个整数 $p \ge 0$ 成立。

### 1. 函数 `is_power_of_2`

只使用除法和取模判断输入是否为 2 的幂（不使用 `math.sqrt`）。

In [13]:
def is_power_of_2(n):
    """判断正整数 n 是否为 2 的幂（只用除法与取模）。"""
    if n < 1:                   # 防御性检查：题目假设输入为正整数
        return False
    while n % 2 == 0:           # 只要还能被 2 整除，就除以 2
        n = n // 2
    return n == 1               # 除到最后剩下 1，说明 n 是 2 的幂

### 2. 函数 `is_power`

把上面的思路推广：`is_power(b, n)` 判断 `n` 是否为 `b` 的幂（参数顺序为 `b`, `n`）。

In [14]:
def is_power(b, n):
    """判断正整数 n 是否为 b 的幂，返回布尔值（只用除法与取模）。"""
    if n < 1 or b < 1:          # 防御性检查：题目假设输入都是正整数
        return False
    if b == 1:                  # 1 的任何次幂都是 1（避免死循环）
        return n == 1
    while n % b == 0:           # 只要还能被 b 整除，就除以 b
        n = n // b
    return n == 1

### 测试

In [15]:
print("is_power_of_2:")
for n in [1, 2, 3, 4, 6, 8, 16, 18, 1024, 2047]:
    print(f"  is_power_of_2({n}) = {is_power_of_2(n)}")

is_power_of_2:
  is_power_of_2(1) = True
  is_power_of_2(2) = True
  is_power_of_2(3) = False
  is_power_of_2(4) = True
  is_power_of_2(6) = False
  is_power_of_2(8) = True
  is_power_of_2(16) = True
  is_power_of_2(18) = False
  is_power_of_2(1024) = True
  is_power_of_2(2047) = False


In [16]:
print("is_power(b, n):")
for b, n in [(2, 8), (2, 10), (3, 1), (3, 9), (3, 27), (3, 28),
             (5, 125), (5, 126), (10, 1000), (10, 1001), (1, 1), (1, 7)]:
    print(f"  is_power({b}, {n}) = {is_power(b, n)}")

is_power(b, n):
  is_power(2, 8) = True
  is_power(2, 10) = False
  is_power(3, 1) = True
  is_power(3, 9) = True
  is_power(3, 27) = True
  is_power(3, 28) = False
  is_power(5, 125) = True
  is_power(5, 126) = False
  is_power(10, 1000) = True
  is_power(10, 1001) = False
  is_power(1, 1) = True
  is_power(1, 7) = False


---

## 小结

- 热身部分实现了 `say_hi`、`goat_pad`、`print_n` 三个简单函数。
- 欧几里得算法：`gcd` 用辗转相除法迭代实现；对负数输入，由于 Python 取模的符号规则，
  当第二个参数为负时结果可能为负，这在数学上不合理，可通过取绝对值修正。
- 欧拉数 e：分别用极限式 (1 + 1/n)^n、级数前 n 项部分和、按误差阈值自适应累加三种方式近似 e。
- 整数属性：用除法与取模实现 `is_power_of_2` 与更一般的 `is_power`。